In [1]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
github_token = secrets.get_secret("github_token")

github_username = "shaambhavi-dubey"
repo_name = "gnn-upi"
repo_url = f"https://{github_token}@github.com/{github_username}/{repo_name}.git"

!git clone {repo_url}
!cd gnn-upi && git config user.email "25bit087@sot.pdpu.ac.in"
!cd gnn-upi && git config user.name "shaambhavi-dubey"

Cloning into 'gnn-upi'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 33 (delta 11), reused 23 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 577.90 KiB | 8.63 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [2]:
# pytorch for graphs
!pip install torch_geometric --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.3 MB/s eta 0:00:00


In [3]:
import pickle
import pandas as pd
import networkx as nx
import numpy as np
import torch

base_path = "/kaggle/input/datasets/shaambhavidubey/gnn-synthetic-data/gnn-upi/data"

with open(f"{base_path}/synthetic_graph.pkl", "rb") as f:
    DirGr = pickle.load(f)

node_df = pd.read_csv(f"{base_path}/node_features.csv")
print(node_df.shape)

(10000, 5)


In [4]:
# we need to convet raw graph into PyG readable format with edge index and feayre matrix which is js one row per node
from torch_geometric.utils import from_networkx

# ttach the features we want as node attributes directly onto the networkx graph first since from_networkx() caries over whatever node attributes exist

for n in DirGr.nodes():
    incoming = [DirGr[u][n]['amount'] for u in DirGr.predecessors(n)]
    outgoing = [DirGr[n][t]['amount'] for t in DirGr.successors(n)]
    total = incoming + outgoing
    DirGr.nodes[n]['avg_amt'] = sum(total)/len(total) if total else 0.0
    DirGr.nodes[n]['max_amount'] = float(max(total)) if total else 0.0
    DirGr.nodes[n]['in_degree'] = DirGr.in_degree(n)
    DirGr.nodes[n]['out_degree'] = DirGr.out_degree(n)

data = from_networkx(DirGr, group_node_attrs=['account_age_days', 'avg_amt', 'max_amount', 'in_degree', 'out_degree'])
print(data)

Data(edge_index=[2, 36905], label=[10000], amount=[36905], x=[10000, 5])


In [5]:
# from_networkx should have carried over 'label' as a graph attribute too — grab it explicitly
labels = torch.tensor([DirGr.nodes[n]['label'] for n in DirGr.nodes()], dtype=torch.long)
data.y = labels

print(data.y.shape, data.y.sum().item())  # should show 10000 total, 250 fraud

torch.Size([10000]) 250


In [6]:
# we need to craete masks because a graph cannot be split into train test sets so we label each node with a mask traint/test/validation
from sklearn.model_selection import train_test_split
import numpy as np

n_nodes = data.num_nodes
indices = np.arange(n_nodes)

train_idx, temp_idx = train_test_split(indices, test_size=0.3, stratify=data.y.numpy(), random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=data.y.numpy()[temp_idx], random_state=42)

train_mask = torch.zeros(n_nodes, dtype=torch.bool)
val_mask = torch.zeros(n_nodes, dtype=torch.bool)
test_mask = torch.zeros(n_nodes, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print(f"Train: {train_mask.sum()}, Val: {val_mask.sum()}, Test: {test_mask.sum()}")
print(f"Train fraud: {data.y[train_mask].sum()}, Val fraud: {data.y[val_mask].sum()}, Test fraud: {data.y[test_mask].sum()}")

Train: 7000, Val: 1500, Test: 1500
Train fraud: 175, Val fraud: 38, Test fraud: 37


In [7]:
#bdefine the gcn model
# change: we increased the capacity and changed from 2 layers to 3
class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv3(x, edge_index)
        return x

NameError: name 'nn' is not defined

In [ ]:
# training part
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)

model = GCN(in_channels=data.x.shape[1], hidden_channels=64, out_channels=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

class_counts = torch.bincount(data.y[data.train_mask])
raw_ratio = (class_counts.sum() / class_counts).float()
class_weights = torch.sqrt(raw_ratio).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def eval_val():
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        probs = F.softmax(out, dim=1)[:, 1]
        val_probs = probs[data.val_mask].cpu().numpy()
        val_labels = data.y[data.val_mask].cpu().numpy()
        return average_precision_score(val_labels, val_probs)

for epoch in range(400):
    loss = train()
    if epoch % 40 == 0:
        val_pr_auc = eval_val()
        print(f"Epoch {epoch}, Loss: {loss:.4f}, Val PR-AUC: {val_pr_auc:.3f}")

print(f"Using device: {device}")
print(f"Class weights: {class_weights}")

In [ ]:
from sklearn.metrics import precision_recall_curve

model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)[:, 1]

val_probs = probs[data.val_mask].cpu().numpy()
val_labels = data.y[data.val_mask].cpu().numpy()

precisions, recalls, thresholds = precision_recall_curve(val_labels, val_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print(f"Best threshold (on val): {best_threshold:.3f}, Val F1 at this threshold: {f1_scores[best_idx]:.3f}")

test_probs = probs[data.test_mask].cpu().numpy()
test_labels = data.y[data.test_mask].cpu().numpy()
test_preds_tuned = (test_probs >= best_threshold).astype(int)

print(classification_report(test_labels, test_preds_tuned))
print(f"F1 (tuned threshold): {f1_score(test_labels, test_preds_tuned):.3f}")
print(f"PR-AUC (threshold-independent): {average_precision_score(test_labels, test_probs):.3f}")

In [ ]:
model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)[:, 1]
    preds = out.argmax(dim=1)

test_preds = preds[data.test_mask].cpu().numpy()
test_labels = data.y[data.test_mask].cpu().numpy()
test_probs = probs[data.test_mask].cpu().numpy()

print(classification_report(test_labels, test_preds))
print(f"PR-AUC: {average_precision_score(test_labels, test_probs):.3f}")
print(f"F1: {f1_score(test_labels, test_preds):.3f}")

In [ ]:
results_04 = {
    "notebook": "04_gcn",
    "architecture": "3-layer GCNConv, hidden=64, dropout=0.5",
    "features_used": ["account_age_days", "avg_amt", "max_amount", "in_degree", "out_degree"],
    "class_weighting": "sqrt(imbalance_ratio)",
    "threshold_tuning": "best F1 threshold selected on validation set",
    "best_threshold": 0.338,
    "precision_fraud": 0.62,
    "recall_fraud": 0.97,
    "f1_fraud": 0.758,
    "pr_auc": 0.623
}

import json
with open("gnn-upi/data/results_04_gcn.json", "w") as f:
    json.dump(results_04, f, indent=2)

print("saved")